# Reconciliation audit: where and why the two sources disagree

Every player-game both sources report is compared on thirteen values (ADR 0005). This notebook
reads the result tables: how much of each season ESPN covers, how often the sources agree, and
which values differ when they do. A rising conflict rate is the first sign one source changed shape.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
print(describe(con))

In [ ]:
summary = con.sql('''
    select league, season, player_games, played_player_games, played_matched,
           round(100 * played_match_rate, 2) as played_match_pct,
           exact, minor, conflict, round(100 * coalesce(conflict_rate, 0), 3) as conflict_pct
    from reconciliation_summary order by league, season
''').df()
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for league, color in (('wnba', '#b23a0a'), ('nba', '#1a7f4b')):
    part = summary[summary['league'] == league]
    axes[0].plot(part['season'], part['played_match_pct'], marker='o', label=league.upper(), color=color)
    axes[1].plot(part['season'], part['conflict_pct'], marker='o', label=league.upper(), color=color)
axes[0].axhline(99, ls='--', color='grey', lw=1); axes[0].set_title('Played player-games matched (%)'); axes[0].legend()
axes[1].set_title('Conflict rate among matched (%)')
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## Which values disagree

`reconciliation` holds one row per differing value with both figures and the resolution
(the league value wins; ESPN fills in only where the league is null). Did-not-play lines and
minutes rounding are left out, so what remains is a genuine disagreement.

In [ ]:
by_stat = con.sql('''
    select stat, count(*) as differences,
           round(avg(abs(delta)), 2) as mean_abs_delta,
           sum(case when resolution = 'espn' then 1 else 0 end) as espn_filled_a_null
    from reconciliation group by 1 order by 2 desc
''').df()
by_stat

In [ ]:
by_team = con.sql('''
    select r.league, r.season, g.team_tricode, count(*) as differences,
           count(distinct r.game_id) as games
    from reconciliation r
    join fct_player_game g on g.league = r.league and g.game_id = r.game_id and g.player_id = r.stats_player_id
    group by 1, 2, 3 order by differences desc limit 15
''').df()
by_team

In [ ]:
sample = con.sql('''
    select r.league, r.season, r.game_id, coalesce(p.full_name, r.stats_player_id) as player,
           r.stat, r.stats_value, r.espn_value, r.delta, r.resolution
    from reconciliation r
    left join (select league, stats_player_id, full_name from dim_player where is_current) p
      on p.league = r.league and p.stats_player_id = r.stats_player_id
    order by abs(r.delta) desc nulls last limit 12
''').df()
sample

## Reading it

Coverage under 99% in a season means ESPN is missing games (its box score returns nothing for a
small share of events) or players the crosswalk could not pair; `player_xwalk_review` in silver
names the players. Conflicts cluster on a few games, which is what a scoring correction by one
source and not the other looks like. The Data Health page shows the same summary table.